# 04. 딜러별 계약출고 관리 — 월타겟 달성 예측용 데이터 탐색

**목표**: 딜러별 월 진행 중(D일 시점) 계약/출고 실적으로 **월말 타겟 달성 여부**를 예측하기 위해 필요한 원천 테이블을 실제로 연결·조회한다.

근거: `simon/ML_실행설계서_10개항목.md` ML① 항목 — 🟢 바로 가능 (일별 실적+타겟 패널, 딜러 21·모델 39, 272개월)

### 필요 테이블
| DB(엔드포인트) | 스키마.테이블 | 용도 |
|---|---|---|
| BP_KTWS → KPI_W | `dbo.OM_CONTRACT` | 딜러별 일별 계약 실적 (CONTRACT_DT, DEALER_ID, MODEL_CD, SFX_CD, CONTRACT_STAT_CD, SOLD_YN, ALLOCATION_YN 등) |
| BP_KTWS → KPI_W | `ktws.FCT_SALES_TARGET_DAILY` | 딜러·모델별 월 타겟 및 일별 타겟(ac_trgt, co_trgt) |
| BP_KTWS → KPI_W | `dbo.VS_SFX` / `dbo.VS_MODEL` | 사양(SFX)·모델 차원 테이블 |
| BP_KTWS → KPI_W | `ktws.DIM_CALENDAR` | 영업일 캘린더 (요일/영업일 보정용) |

> 참고: 과거 `VT_DAILY_SALES_TREND`는 갱신이 멈춘 legacy 테이블이므로 사용하지 않고, 실적은 `OM_CONTRACT`에서 직접 파생한다.

## 0. 준비

In [23]:
import pyodbc
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

# BP/KTWS 엔드포인트 (KPI_W: OM_CONTRACT, FCT_SALES_TARGET_DAILY 등이 위치한 DB)
BPKTWS_SERVER = "6orm62c43rguff2mpdxwqy76tu-xhehnpiu2bautgglhximxpyqca.datawarehouse.fabric.microsoft.com"
DATABASE = "KPI_W"

def make_conn(server, database=None):
    db_part = f"DATABASE={database};" if database else ""
    conn_str = f"""
    DRIVER={{ODBC Driver 17 for SQL Server}};
    SERVER={server},1433;
    {db_part}
    Encrypt=yes;
    TrustServerCertificate=no;
    Authentication=ActiveDirectoryInteractive;
    Connection Timeout=30;
    """
    return pyodbc.connect(conn_str)

conn = make_conn(BPKTWS_SERVER, DATABASE)
pd.read_sql("SELECT DB_NAME() AS current_db", conn)

C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_34588\2346834732.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("SELECT DB_NAME() AS current_db", conn)


,current_db
0,KPI_W


## 1. 딜러별 일별 계약 실적 — `dbo.OM_CONTRACT`

월말 달성 예측의 타깃(라벨) 및 실적 피처를 만드는 핵심 원천. 계약일 기준 최근 데이터를 확인한다.

In [24]:
om_contract = pd.read_sql("""
SELECT *
FROM dbo.OM_CONTRACT
ORDER BY contract_dt DESC
""", conn)

print(om_contract.shape)
om_contract.head(5)

C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_34588\1583621790.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  om_contract = pd.read_sql("""


(410947, 134)


,contract_no,dlr_contract_no,contract_dt,contract_stat_cd,sold_yn,urgent_yn,allocation_yn,status_mod_dt,cond_mod_yn,dealer_id,shop_cd,user_id,owner_type,cust_seq,comp_seq,real_cust_seq,owner_seq,customs_seq,brand_cd,model_cd,variant_cd,my_cd,sfx_cd,col_combi_cd,option_cd,option_cd2,option_cd3,option_cd4,vin,vehic_magic,vehic_price,vehic_vat,vehic_option_price,vehic_color_price,vehic_discount_amt,vehic_total_amt,deposit_amt,sales_type,pay_type,tax_type,lease_comp_seq,reg_free_yn,reg_stock_free_yn,reg_stock_rate,reg_stock_buy_yn,reg_agency_yn,reg_tax,reg_stock_price,reg_stamp_price,reg_plate_price,reg_fee,reg_aquisition_tax,reg_special_tax,reg_education_tax,reg_total_amt,take_contract_amt,take_delivery_amt,lease_month_amt,lease_term_dt,lease_rate,take_depositer_nm,take_deposit_cd,side_stamp_price,side_setup_amt,side_fee,side_total_amt,delivery_place_cd,delivery_plan2_dt,delivery_delay_reason,delivery_actual_dt,delivery_actual_hour,delivery_plate_cd,request_by,request_dt,approval_by,approval_dt,cancel_by,cancel_dt,last_retail_sales_dt,last_issued_dt,last_mod_dt,delivery_request_by,delivery_request_dt,delivery_cancel_by,delivery_cancel_dt,delivery_plan_by,delivery_plan_dt,delivery_approval_by,delivery_approval_dt,reg_plan_dt,contract_msg,vehic_reg_no,vehic_reg_dt,dept_cd,boc_except_dt,reg_dt,reg_user_id,upd_dt,upd_user_id,public_yn,allocation_dt,prev_contract_stat_cd,rs_cust_zip,rs_cust_addr,rs_cust_addr2,rs_geo_loc_x,rs_geo_loc_y,potential_division,org_followup_id,plate_size,receiver_apply_yn,fiber_use_yn,if_send_yn,recept_no,receiver_ssn,pma_yn,cust_taxpay_no,family_seq,lemon_yn,lemon_yn_choice,app_flag,consign_sales_flag,contract_msg_kr,cust_ci_chk_yn,cust_ci_chk_except_yn,realnm_seq,tax_biz_no,pma_city,pma_gu,taxpay_no_ymd,taxpay_no_g,flood_yn,ELT_TIMESTAMP,BRAND
0,311342.0,CW260212-148,2026-02-12,D4,Y,N,Y,2026-02-19 10:08:53,N,CW00000,CW00000,CW1711,02,1806868.0,2266383.0,2266383.0,513484.0,NaN,L,RX,AALH15L-AWXGBA,2026,RJ,223-21,NaN,None,None,None,JTJCJAJA6T2068025,6096510.0,84481818.0,8448182.0,0.0,0.0,4500000.0,92930000.0,98938720.0,96,05,10017.0,513484.0,N,N,0.0,N,Y,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1000000.0,0.0,0.0,2026-02-12,0.0,NaN,01,0.0,0.0,0.0,0.0,01,NaN,None,2026-02-20,NaN,02,CW1711,2026-02-12 10:13:18.0000000,CW1043,2026-02-19 10:08:32.0000000,NaN,NaN,2026-02-19 10:11:24,2026-02-19 10:11:24,NaN,CW1043,2026-02-19 10:08:49.0000000,NaN,NaN,NaN,2026-02-19,CW1043,2026-02-19 10:08:54.0000000,2026-02-19,"키지갑, 골프백세트 or 여행용세트중 여행용세트 선택",157나1301,2026-02-19,CW20103,2026-02-12 10:13:17,2026-02-12 10:13:18.0000000,CW1711,2026-02-19 17:30:41.0000000,CW1711,0,2026-02-19 10:08:44.0000000,NaN,10598,경기 고양시 덕양구 동산2로 31,301호 (동산동),NaN,NaN,NaN,NaN,02,0,0,Y,L2026020653,0y57wFWhY7JJUC+kKaKr8w==,N,0y57wFWhY7JJUC+kKaKr8w==,0.0,Y,Y,Y,N,None,Y,N,103350.0,NaN,None,None,NaN,NaN,N,2026-02-22T07:16:50.835431Z,LEXUS
1,311231.0,CW260211-124,2026-02-11,D4,Y,N,Y,2026-02-20 10:56:48,N,CW00000,CW00000,CW1264,02,2266190.0,2266189.0,2266189.0,1860591.0,NaN,L,NX,AAZH26L-AWXLBA,2026,VF,223-43,NaN,None,None,None,JTJKKAFZ7T2088517,6097739.0,73918182.0,7391818.0,0.0,0.0,1000000.0,81310000.0,85084270.0,21,05,10017.0,1860591.0,N,N,0.0,N,Y,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,500000.0,0.0,0.0,2029-02-20,0.0,NaN,03,0.0,0.0,0.0,0.0,01,NaN,None,2026-02-20,NaN,01,CW1264,2026-02-11 09:24:46.0000000,CW1043,2026-02-20 10:56:25.0000000,NaN,NaN,2026-02-20 10:57:15,2026-02-20 10:57:15,NaN,CW1043,2026-02-20 10:56:42.0000000,NaN,NaN,NaN,2026-02-20,CW1043,2026-02-20 10:56:49.0000000,2026-02-20,"키지갑 , 골프백세트 or 여행용세트중 골프백세트 선택",171라2310,2026-02-20,CW20103,2026-02-11 09:24:45,2026-02-11 09:24:46.0000000,CW1264,2026-02-20 16:37:00.0000000,CW1264,0,2026-02-20 10:56:36.0000000,NaN,06284,서울특별시 강남구,삼성로 212 은마아파트 302호,NaN,NaN,NaN,NaN,02,0,0,Y,L2026020782,qwNILVJY1AIxt3dh6DaJ2A==,N,qwNILVJY1AIxt3dh6DaJ2A==,0.0,Y,Y,NaN,N,None,Y,N,103151.0,NaN,None,None,NaN,NaN,NaN,2026-02-22T07:16:50.835431Z,LEXUS
2,298629.0,JM260210-020,2026-02-10,D4,Y,N,Y,2026-02-19 13:44:34,N,JM00000,JM00000,JM3

In [3]:
# 딜러 수 / 모델 수 / 계약일 범위 — 실측 근거(ML 설계서 상 딜러 21·모델 39와 비교)
pd.read_sql("""
SELECT
    COUNT(DISTINCT dealer_id) AS dealer_cnt,
    COUNT(DISTINCT model_cd)  AS model_cnt,
    MIN(contract_dt) AS min_contract_dt,
    MAX(contract_dt) AS max_contract_dt,
    COUNT(*) AS row_cnt
FROM dbo.OM_CONTRACT
""", conn)

C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_31904\2832888898.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("""


,dealer_cnt,model_cnt,min_contract_dt,max_contract_dt,row_cnt
0,20,35,2001-01-02,2026-02-12,410997


In [9]:
om_contract["dealer_id"].value_counts()

dealer_id
DT00000    39991
PM00000    36824
CW00000    35528
CT00000    32618
HS00000    31976
KJ00000    30160
TD00000    28807
LS00000    28561
KM00000    25330
YM00000    20439
DM00000    19767
TY00000    15805
SY00000    15004
NY00000    12093
JB00000    10769
TI00000     9755
PH00000     7698
JM00000     6129
SK00000     2263
TM00000     1430
Name: count, dtype: int64

## 2. 딜러·모델별 일별/월별 타겟 — `ktws.FCT_SALES_TARGET_DAILY`

`ac_trgt`(월 누적 타겟), `co_trgt`, `ac_trgt_daily`/`co_trgt_daily`(일별 배분 타겟)를 확인한다.

In [25]:
sales_target = pd.read_sql("""
SELECT *
FROM ktws.FCT_SALES_TARGET_DAILY
ORDER BY dealer_key DESC
""", conn)

print(sales_target.shape)
sales_target.head(5)

C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_34588\408696888.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sales_target = pd.read_sql("""


(96632, 11)


,dealer_key,brand_cd,model_key,mon_target_variant,dateby,ac_trgt,co_trgt,ac_trgt_daily,co_trgt_daily,BRAND,ETL_TIMESTAMP
0,TOYOTATY00000,T,TOYOTACRW,CROWN-HEV LTD,2026-01-05,NaN,2.0,0.0,1.0,TOYOTA,2026-07-07 14:49:39.493333
1,TOYOTATY00000,T,TOYOTACAM,CAMRY(N3)-HV,2026-01-23,8.0,15.0,1.0,1.0,TOYOTA,2026-07-07 14:49:39.493333
2,TOYOTATY00000,T,TOYOTARAV,RAV4-PHEV,2025-03-21,1.0,2.0,0.0,0.0,TOYOTA,2026-07-07 14:49:39.493333
3,TOYOTATY00000,T,TOYOTAT86,86 MT 26Y L,2026-04-26,1.0,1.0,0.0,0.0,TOYOTA,2026-07-07 14:49:39.493333
4,TOYOTATY00000,T,TOYOTARAV,RAV4-HV AWD(N),2025-04-27,9.0,13.0,0.0,1.0,TOYOTA,2026-07-07 14:49:39.493333


In [5]:
# 타겟 결측률 확인 (ML 설계서: ac_trgt 약 28% 결측으로 보고됨)
pd.read_sql("""
SELECT
    COUNT(*) AS row_cnt,
    SUM(CASE WHEN ac_trgt IS NULL THEN 1 ELSE 0 END) AS ac_trgt_null_cnt,
    MIN(dateby) AS min_dateby,
    MAX(dateby) AS max_dateby
FROM ktws.FCT_SALES_TARGET_DAILY
""", conn)

C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_31904\3133359077.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("""


,row_cnt,ac_trgt_null_cnt,min_dateby,max_dateby
0,91089,25392,2025-01-01,2026-06-30


### 2-1. ac_trgt / ac_trgt_daily 의미 검증

DB 정의서에는 `ac_trgt`, `co_trgt`, `ac_trgt_daily`, `co_trgt_daily`의 정확한 의미가 정의돼 있지 않다(추정 의미 칸 공란).
`AC = 계약(Contract)`, `CO = 출고(Delivery)`라는 가설을 세웠는데, 같은 딜러·모델·세부모델(mon_target_variant)을 고정하고
`dateby`별로 값이 어떻게 변하는지 직접 찍어보면 다음을 확인할 수 있다.

- `ac_trgt`가 한 달 내내 **동일한 값으로 반복**되는지(= 월 목표를 매일 복제 저장) vs **날짜가 지날수록 줄어드는지**(= 잔여 목표)
- `ac_trgt_daily`를 한 달치 **합산하면 ac_trgt와 비슷해지는지**(= 월 목표를 일별로 쪼갠 배분값이라는 뜻)
- `ac_*` 계열과 `co_*` 계열이 서로 다른 값 패턴을 보이는지(다른 계열 = 별개 지표라는 근거)

In [26]:

# 실제 데이터에 존재하는 조합 하나를 고정 (2026-06-30 조회 결과 중 ac_trgt=50인 LEXUSCT00000 / LEXUSNX / NX350h 사용)
DEALER_KEY = 'LEXUSCT00000'
MODEL_KEY = 'LEXUSNX'
VARIANT = 'NX350h'

trgt_trace = pd.read_sql(f"""
SELECT dateby, ac_trgt, co_trgt, ac_trgt_daily, co_trgt_daily
FROM ktws.FCT_SALES_TARGET_DAILY
WHERE dealer_key = '{DEALER_KEY}'
  AND model_key = '{MODEL_KEY}'
  AND mon_target_variant = '{VARIANT}'
ORDER BY dateby
""", conn)

print(trgt_trace.shape)
trgt_trace

C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_34588\3900893544.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  trgt_trace = pd.read_sql(f"""


(574, 5)


,dateby,ac_trgt,co_trgt,ac_trgt_daily,co_trgt_daily
0,2025-01-01,NaN,1.0,0.0,1.0
1,2025-01-02,NaN,3.0,0.0,2.0
2,2025-01-03,NaN,5.0,0.0,2.0
3,2025-01-04,NaN,8.0,0.0,3.0
4,2025-01-05,NaN,11.0,0.0,3.0
...,...,...,...,...,...
569,2026-07-27,28.0,46.0,3.0,2.0
570,2026-07-28,30.0,47.0,2.0,1.0
571,2026-07-29,32.0,49.0,2.0,2.0
572,2026-07-30,34.0,51.0,2.0,2.0


In [27]:
# 패턴 요약: 월 전체를 대상으로 ac_trgt가 고정값인지, ac_trgt_daily의 합이 ac_trgt와 맞아떨어지는지 확인
trgt_trace['yyyymm'] = trgt_trace['dateby'].astype(str).str.slice(0, 7)

summary = trgt_trace.groupby('yyyymm').agg(
    days=('dateby', 'count'),                      # 해당 월에 조회된 일(row) 수
    ac_trgt_nunique=('ac_trgt', 'nunique'),         # 그 달 ac_trgt가 가진 서로 다른 값의 개수 — 1이면 월 내내 고정값(월 목표를 매일 복제), 여러 개면 날짜별로 값이 변함(=누적치일 가능성)
    ac_trgt_first=('ac_trgt', 'first'),             # 그 달 1일(또는 첫 조회일)의 ac_trgt 값
    ac_trgt_last=('ac_trgt', 'last'),                # 그 달 말일(또는 마지막 조회일)의 ac_trgt 값 — first와 다르면 누적치라는 뜻
    ac_trgt_daily_sum=('ac_trgt_daily', 'sum'),     # 그 달 ac_trgt_daily(일별 증분)를 전부 합한 값 — 이 값이 ac_trgt_last와 비슷하면 "daily가 누적치의 일별 배분값"이라는 가설이 맞다는 근거
    co_trgt_nunique=('co_trgt', 'nunique'),         # co_trgt 버전의 nunique (위 ac_trgt_nunique와 동일한 의미)
    co_trgt_first=('co_trgt', 'first'),             # co_trgt 버전의 first
    co_trgt_last=('co_trgt', 'last'),                # co_trgt 버전의 last
    co_trgt_daily_sum=('co_trgt_daily', 'sum'),     # co_trgt 버전의 daily 합계
)
summary

,days,ac_trgt_nunique,ac_trgt_first,ac_trgt_last,ac_trgt_daily_sum,co_trgt_nunique,co_trgt_first,co_trgt_last,co_trgt_daily_sum
yyyymm,,,,,,,,,
2025-01,31,15,2.0,40.0,40.0,31,1.0,45.0,45.0
2025-02,28,18,1.0,39.0,39.0,28,2.0,42.0,42.0
2025-03,28,18,1.0,44.0,44.0,28,6.0,58.0,58.0
2025-04,30,20,1.0,45.0,45.0,30,3.0,50.0,50.0
2025-05,31,17,1.0,43.0,43.0,31,2.0,52.0,52.0
2025-06,30,18,1.0,43.0,43.0,30,1.0,54.0,54.0
2025-07,31,20,1.0,41.0,41.0,31,4.0,59.0,59.0
2025-08,31,19,1.0,47.0,47.0,31,3.0,56.0,56.0
2025-09,30,18,1.0,31.0,31.0,30,3.0,48.0,48.0


## 3. 차원 테이블 — `dbo.VS_SFX`, `dbo.VS_MODEL`, `ktws.DIM_CALENDAR`

In [ ]:
vs_model = pd.read_sql("SELECT TOP 200 * FROM dbo.VS_MODEL", conn)
vs_sfx = pd.read_sql("SELECT TOP 200 * FROM dbo.VS_SFX", conn)
dim_calendar = pd.read_sql("SELECT TOP 200 * FROM ktws.DIM_CALENDAR ORDER BY 1 DESC", conn)

print("VS_MODEL:", vs_model.shape)
print("VS_SFX:", vs_sfx.shape)
print("DIM_CALENDAR:", dim_calendar.shape)
vs_model.head()

C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_34588\2240321022.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  vs_model = pd.read_sql("SELECT TOP 200 * FROM dbo.VS_MODEL", conn)
C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_34588\2240321022.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  vs_sfx = pd.read_sql("SELECT TOP 200 * FROM dbo.VS_SFX", conn)
C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_34588\2240321022.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_calendar = pd.read_sql("SELECT T

VS_MODEL: (135, 11)
VS_SFX: (200, 41)
DIM_CALENDAR: (200, 16)


,brand_cd,model_cd,model_nm,curr_sales_yn,display_order,reg_dt,upd_dt,reg_user_id,upd_user_id,ELT_TIMESTAMP,BRAND
0,L,LS,LS,Y,1.0,2011-01-02 02:33:16.0000000,2011-01-02 02:33:16.0000000,ADMIN,2222222222222,2026-07-07T14:11:57.9635415Z,LEXUS
1,L,GS,GS,Y,5.0,2011-01-02 02:33:16.0000000,2019-11-27 10:06:56.0000000,ADMIN,TMR040161,2026-07-07T14:11:57.9635415Z,LEXUS
2,L,ES,ES,Y,10.0,2011-01-02 02:33:16.0000000,2011-01-02 02:33:16.0000000,ADMIN,ADMIN,2026-07-07T14:11:57.9635415Z,LEXUS
3,L,IS,IS,Y,15.0,2011-01-02 02:33:16.0000000,2011-01-02 02:33:16.0000000,ADMIN,ADMIN,2026-07-07T14:11:57.9635415Z,LEXUS
4,L,RX,RX,Y,20.0,2011-01-02 02:33:16.0000000,2011-01-02 02:33:16.0000000,ADMIN,2222222222222,2026-07-07T14:11:57.9635415Z,LEXUS


In [7]:
vs_sfx.head()

,brand_cd,model_cd,variant_cd,my_cd,sfx_cd,brand_nm,model_nm,variant_nm,model_year,sfx_nm,curr_sales_yn,display_order,launch_dt,form_apply,motive_type,taking_fix,displacement,hp,gross_weight,cylinder_cnt,max_load,max_output,length,width,height,order_yn,reg_dt,upd_dt,hybrid_yn,navi_yn,ecfrnd_vhcle_knd,grade,connected_car_yn,spec_variant_nm,hi_pass_yn,black_box_yn,consign_sales_flag,ew_yn,dcm_type,ELT_TIMESTAMP,BRAND
0,L,CT,ZWA10L-AHXBBA,2020,CY,LEXUS,CT,CT200h,2020,CY,N,2.0,20190424,010-2-00034-0012-1219,2ZR,5,1798,99,1780,4.0,0,136.0,4355.0,1765.0,1455.0,N,2019-02-25 18:42:34.0000000,2023-08-11 10:14:43.0000000,Y,N,2,Supreme,None,렉서스 CT200h,None,None,N,N,None,2026-07-03T13:13:02.0023861Z,LEXUS
1,T,T86,ZN6-FYE7,2018,8H,Toyota,TOYOTA 86,86 AT,2018,8H,N,11.0,20180201,010-2-00042-0007-1217,FA20,4,1998,203,1540,4.0,0,203.0,4240.0,1775.0,1320.0,N,2017-09-29 19:40:23.0000000,2023-03-03 10:30:38.0000000,N,N,NaN,AT,None,토요타 86 AT,None,None,N,N,None,2026-07-03T13:11:19.234407Z,TOYOTA
2,T,T86,ZN6-GYE8,2019,8M,Toyota,TOYOTA 86,86 MT,2019,8M,N,17.0,20180701,010-2-00043-0008-1218,FA20,4,1998,203,1500,4.0,0,203.0,4240.0,1775.0,1320.0,N,2018-05-03 13:58:14.0000000,2023-03-03 10:30:38.0000000,N,N,NaN,MT,None,토요타 86 MT,None,None,N,N,None,2026-07-03T13:11:19.234407Z,TOYOTA
3,L,IS,GXE10L-AEPVKW,2000,IS,LEXUS,IS,IS200,2000,IS,N,1.0,20020801,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,2006-03-15 00:00:00.0000000,2017-05-23 17:18:42.0000000,N,N,NaN,NaN,None,렉서스 IS200,None,None,NaN,N,None,2026-07-03T13:13:02.0023861Z,LEXUS
4,L,GS,UZS190L-BETQKA,2008,GP,LEXUS,GS,GS430,2008,GP,N,0.0,20071101,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,0.0,0.0,N,2007-06-11 09:36:00.0000000,2017-05-23 17:18:42.0000000,N,N,NaN,NaN,None,렉서스 GS430,None,None,NaN,N,None,2026-07-03T13:13:02.0023861Z,LEXUS


In [8]:
dim_calendar.head()

,Date,Year,QuarterNumber,MonthNumber,Day,DayName,WeekNumber,WeekNumber_Monthly,WeekNumber_Monthly_txt,QuarterText,MonthName,MonthAbbr,WeekdayName,WeekdayAbbr,YearMonth,WeekdayNumber
0,2026-08-31,2026,3,8,31,31일,36,4,4주차,Q3,8월,8월,월요일,월,2026-08,1
1,2026-08-30,2026,3,8,30,30일,36,4,4주차,Q3,8월,8월,일요일,일,2026-08,7
2,2026-08-29,2026,3,8,29,29일,35,4,4주차,Q3,8월,8월,토요일,토,2026-08,6
3,2026-08-28,2026,3,8,28,28일,35,4,4주차,Q3,8월,8월,금요일,금,2026-08,5
4,2026-08-27,2026,3,8,27,27일,35,4,4주차,Q3,8월,8월,목요일,목,2026-08,4


## 4. 딜러×월 실적 vs 타겟 결합 미리보기

`OM_CONTRACT`를 딜러×월로 집계한 실적과 `FCT_SALES_TARGET_DAILY`의 월 타겟(`ac_trgt`)을 붙여 달성률을 미리 확인한다.

> 주의: `contract_dt`가 문자열(varchar)이므로 정제 후 사용. `eod_month=44652` 등 더티 날짜값은 제외해야 한다.

In [19]:
# 주의: contract_dt는 'YYYY-MM-DD' 형식이므로 LEFT(contract_dt,6)='YYYY-M'이 되어 버그였음 -> REPLACE 후 LEFT(7)로 수정.
# 타겟 테이블과 조인하려면 dealer_id뿐 아니라 model_cd까지 맞춰야 하므로 model_cd도 함께 집계.
# 계약(contract_dt 기준)과 별도로, 출고(delivery_actual_dt 기준) 실적도 같은 방식으로 집계한다.
dealer_model_month_actual = pd.read_sql("""
SELECT
    dealer_id,
    model_cd,
    REPLACE(LEFT(contract_dt, 7), '-', '') AS yyyymm,
    COUNT(*) AS contract_cnt,
    SUM(CASE WHEN sold_yn = 'Y' THEN 1 ELSE 0 END) AS sold_cnt
FROM dbo.OM_CONTRACT
WHERE contract_dt LIKE '2[0-9][0-9][0-9]-%'  -- 더티 날짜값 제외
GROUP BY dealer_id, model_cd, LEFT(contract_dt, 7)
ORDER BY yyyymm DESC, dealer_id, model_cd
""", conn)

dealer_model_month_delivery = pd.read_sql("""
SELECT
    dealer_id,
    model_cd,
    REPLACE(LEFT(delivery_actual_dt, 7), '-', '') AS yyyymm,
    COUNT(*) AS delivery_cnt
FROM dbo.OM_CONTRACT
WHERE delivery_actual_dt LIKE '2[0-9][0-9][0-9]-%'
GROUP BY dealer_id, model_cd, LEFT(delivery_actual_dt, 7)
ORDER BY yyyymm DESC, dealer_id, model_cd
""", conn)

print(dealer_model_month_actual.shape, dealer_model_month_delivery.shape)
dealer_model_month_actual.head(20)

C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_34588\3902681792.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dealer_model_month_actual = pd.read_sql("""
C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_34588\3902681792.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dealer_model_month_delivery = pd.read_sql("""


(22133, 5) (20881, 4)


,dealer_id,model_cd,yyyymm,contract_cnt,sold_cnt
0,CT00000,ES,202602,2,2
1,CT00000,NX,202602,3,1
2,CT00000,RX,202602,1,0
3,CT00000,UX,202602,1,0
4,CW00000,ES,202602,3,3
5,CW00000,NX,202602,1,1
6,CW00000,RX,202602,1,1
7,DM00000,APD,202602,1,1
8,DM00000,RAV,202602,1,0
9,DT00000,NX,202602,2,1


In [20]:
# 주의: dealer_key='TOYOTATY00000'처럼 브랜드명 문자열 + dealer_id 이고, model_key도 브랜드명 + model_cd 이다.
# OM_CONTRACT의 dealer_id/model_cd와 맞추려면 앞의 브랜드명(BRAND 컬럼 값)을 SUBSTRING으로 잘라내야 한다(이전 버전은 이 처리가 없어 조인이 전부 실패했음).
# 월 총 타겟은 ac_trgt(누적치)의 월말값이 아니라 ac_trgt_daily/co_trgt_daily의 월 합계로 구한다 (2-1 검증 결과: *_trgt는 월중 누적값, *_trgt_daily가 그 증분).
dealer_model_month_target = pd.read_sql("""
SELECT
    SUBSTRING(dealer_key, LEN(BRAND) + 1, 50) AS dealer_id,
    SUBSTRING(model_key, LEN(BRAND) + 1, 50) AS model_cd,
    FORMAT(dateby, 'yyyyMM') AS yyyymm,
    SUM(ac_trgt_daily) AS ac_trgt_daily_sum,
    SUM(co_trgt_daily) AS co_trgt_daily_sum
FROM ktws.FCT_SALES_TARGET_DAILY
GROUP BY SUBSTRING(dealer_key, LEN(BRAND) + 1, 50), SUBSTRING(model_key, LEN(BRAND) + 1, 50), FORMAT(dateby, 'yyyyMM')
ORDER BY yyyymm DESC, dealer_id, model_cd
""", conn)

print(dealer_model_month_target.shape)
dealer_model_month_target.head(20)

C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_34588\2858123072.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dealer_model_month_target = pd.read_sql("""


(2221, 5)


,dealer_id,model_cd,yyyymm,ac_trgt_daily_sum,co_trgt_daily_sum
0,CT00000,ES,202607,77.0,57.0
1,CT00000,LM,202607,11.0,15.0
2,CT00000,LS,202607,0.0,1.0
3,CT00000,LX,202607,5.0,6.0
4,CT00000,NX,202607,53.0,68.0
5,CT00000,RX,202607,29.0,38.0
6,CT00000,UX,202607,21.0,16.0
7,CW00000,ES,202607,78.0,58.0
8,CW00000,LM,202607,14.0,20.0
9,CW00000,LS,202607,1.0,2.0


In [21]:
achievement_preview = (
    dealer_model_month_actual
    .merge(dealer_model_month_delivery, on=['dealer_id', 'model_cd', 'yyyymm'], how='left')
    .merge(dealer_model_month_target, on=['dealer_id', 'model_cd', 'yyyymm'], how='left')
)
achievement_preview['delivery_cnt'] = achievement_preview['delivery_cnt'].fillna(0)

# 두 가지 짝짓기 가설을 모두 계산해서 비교한다.
achievement_preview['contract_vs_ac'] = achievement_preview['contract_cnt'] / achievement_preview['ac_trgt_daily_sum']
achievement_preview['contract_vs_co'] = achievement_preview['contract_cnt'] / achievement_preview['co_trgt_daily_sum']
achievement_preview['delivery_vs_ac'] = achievement_preview['delivery_cnt'] / achievement_preview['ac_trgt_daily_sum']
achievement_preview['delivery_vs_co'] = achievement_preview['delivery_cnt'] / achievement_preview['co_trgt_daily_sum']

achievement_preview.sort_values('yyyymm', ascending=False).head(30)

,dealer_id,model_cd,yyyymm,contract_cnt,sold_cnt,delivery_cnt,ac_trgt_daily_sum,co_trgt_daily_sum,contract_vs_ac,contract_vs_co,delivery_vs_ac,delivery_vs_co
0,CT00000,ES,202602,2,2,3.0,62.0,71.0,0.032258,0.028169,0.048387,0.042254
1,CT00000,NX,202602,3,1,3.0,51.0,70.0,0.058824,0.042857,0.058824,0.042857
2,CT00000,RX,202602,1,0,5.0,31.0,31.0,0.032258,0.032258,0.161290,0.161290
3,CT00000,UX,202602,1,0,1.0,8.0,13.0,0.125000,0.076923,0.125000,0.076923
4,CW00000,ES,202602,3,3,6.0,70.0,80.0,0.042857,0.037500,0.085714,0.075000
5,CW00000,NX,202602,1,1,2.0,59.0,80.0,0.016949,0.012500,0.033898,0.025000
6,CW00000,RX,202602,1,1,2.0,35.0,36.0,0.028571,0.027778,0.057143,0.055556
7,DM00000,APD,202602,1,1,1.0,12.0,14.0,0.083333,0.071429,0.083333,0.071429
8,DM00000,RAV,202602,1,0,1.0,18.0,15.0,0.055556,0.066667,0.055556,0.066667
9,DT00000,NX,202602,2,1,2.0,93.0,128.0,0.021505,0.015625,0.021505,0.015625


## 5. AC / CO 타겟이 계약(contract)·출고(delivery) 중 무엇인지 검증

월 단위 집계만으로는 판별이 어렵다 — 한 딜러·모델의 계약 실적과 출고 실적은 거의 같은 규모로 움직이기 때문에(계약된 차는 대부분 그 근방에 출고됨),
`contract_cnt`와 `delivery_cnt`의 상관관계가 0.98 이상으로 서로 거의 동일한 신호가 되어 월 합계 비교로는 AC/CO를 구분할 수 없다.

대신 **일별 타이밍(day-level timing)**으로 구분한다:
1. `ac_trgt_daily`/`co_trgt_daily`와 `contract_cnt`/`delivery_cnt`를 dealer_id·model_cd·일(day) 단위로 맞춰 시차(lag)를 -10~+10일로 밀어가며 상관관계를 스캔 — 어느 실적 계열이 lag=0(당일)에서 뚜렷한 상관 피크를 만드는지 확인.
2. 월중 날짜 위치(초순/중순/말일)별 평균값 비율(말일÷초순)을 비교 — 자동차 업계 특유의 "월말 출고 밀어내기"(month-end delivery push) 패턴이 있다면 출고 실적과 그 타겟 계열만 말일에 몰릴 것.

In [28]:
contract_daily = pd.read_sql("""
SELECT dealer_id, model_cd, TRY_CAST(contract_dt AS date) AS ymd, COUNT(*) AS contract_cnt
FROM dbo.OM_CONTRACT
WHERE TRY_CAST(contract_dt AS date) IS NOT NULL
GROUP BY dealer_id, model_cd, TRY_CAST(contract_dt AS date)
""", conn)

delivery_daily = pd.read_sql("""
SELECT dealer_id, model_cd, TRY_CAST(delivery_actual_dt AS date) AS ymd, COUNT(*) AS delivery_cnt
FROM dbo.OM_CONTRACT
WHERE TRY_CAST(delivery_actual_dt AS date) IS NOT NULL
GROUP BY dealer_id, model_cd, TRY_CAST(delivery_actual_dt AS date)
""", conn)

target_daily = pd.read_sql("""
SELECT
    SUBSTRING(dealer_key, LEN(BRAND) + 1, 50) AS dealer_id,
    SUBSTRING(model_key, LEN(BRAND) + 1, 50) AS model_cd,
    TRY_CAST(dateby AS date) AS ymd,
    SUM(ac_trgt_daily) AS ac_trgt_daily,
    SUM(co_trgt_daily) AS co_trgt_daily
FROM ktws.FCT_SALES_TARGET_DAILY
WHERE TRY_CAST(dateby AS date) IS NOT NULL
GROUP BY SUBSTRING(dealer_key, LEN(BRAND) + 1, 50), SUBSTRING(model_key, LEN(BRAND) + 1, 50), TRY_CAST(dateby AS date)
""", conn)

for df in (contract_daily, delivery_daily, target_daily):
    df['ymd'] = pd.to_datetime(df['ymd'])

# 실적이 실제로 존재하는 기간(=contract_dt 최댓값)까지만 비교 — 그 이후는 미래 타겟만 있고 실적이 없어 비교 불가.
max_actual_dt = contract_daily['ymd'].max()

daily = (
    target_daily[target_daily['ymd'] <= max_actual_dt]
    .merge(contract_daily, on=['dealer_id', 'model_cd', 'ymd'], how='left')
    .merge(delivery_daily, on=['dealer_id', 'model_cd', 'ymd'], how='left')
)
daily[['contract_cnt', 'delivery_cnt']] = daily[['contract_cnt', 'delivery_cnt']].fillna(0)

print(daily.shape)
daily[['ac_trgt_daily', 'co_trgt_daily', 'contract_cnt', 'delivery_cnt']].corr()

C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_34588\2660511384.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  contract_daily = pd.read_sql("""
C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_34588\2660511384.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  delivery_daily = pd.read_sql("""
C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_34588\2660511384.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  target_daily = pd.read_sql("""


(44003, 7)


,ac_trgt_daily,co_trgt_daily,contract_cnt,delivery_cnt
ac_trgt_daily,1.000000,0.427174,0.212475,0.351519
co_trgt_daily,0.427174,1.000000,0.330689,0.279752
contract_cnt,0.212475,0.330689,1.000000,0.637367
delivery_cnt,0.351519,0.279752,0.637367,1.000000


In [29]:
# 딜러·모델 단위 노이즈를 줄이기 위해 전사 일별 합계로 집계한 뒤, -10~+10일 시차로 상관관계를 스캔한다.
daily_sys = daily.groupby('ymd')[['ac_trgt_daily', 'co_trgt_daily', 'contract_cnt', 'delivery_cnt']].sum().sort_index()

lag_scan = []
for lag in range(-10, 11):
    lag_scan.append({
        'lag': lag,
        'ac_vs_contract': daily_sys['ac_trgt_daily'].corr(daily_sys['contract_cnt'].shift(lag)),
        'ac_vs_delivery': daily_sys['ac_trgt_daily'].corr(daily_sys['delivery_cnt'].shift(lag)),
        'co_vs_contract': daily_sys['co_trgt_daily'].corr(daily_sys['contract_cnt'].shift(lag)),
        'co_vs_delivery': daily_sys['co_trgt_daily'].corr(daily_sys['delivery_cnt'].shift(lag)),
    })
lag_scan = pd.DataFrame(lag_scan).set_index('lag')
print(lag_scan.round(3))
print()
print('각 계열의 최대 상관계수가 나타나는 lag:')
print(lag_scan.abs().idxmax())

     ac_vs_contract  ac_vs_delivery  co_vs_contract  co_vs_delivery
lag                                                                
-10          -0.054          -0.080           0.039           0.075
-9           -0.049          -0.076           0.018           0.071
-8            0.022           0.061           0.027           0.021
-7           -0.006           0.167           0.109           0.061
-6           -0.030           0.014           0.044           0.033
-5            0.021          -0.055           0.031           0.008
-4            0.006          -0.030           0.070           0.062
-3           -0.025           0.024           0.038           0.049
-2           -0.022           0.052           0.058           0.069
-1            0.058           0.221           0.081           0.002
0             0.065           0.436           0.246           0.048
1             0.052           0.086           0.116           0.195
2             0.068          -0.048           0.

In [ ]:
# 월중 위치(초순/중순/말일)별 평균값 비교 — "말일/초순" 비율이 1보다 크면 월말에 쏠리는(=출고 밀어내기형) 계열.
tmp = daily.copy()
tmp['dom_ratio'] = tmp['ymd'].dt.day / tmp['ymd'].dt.days_in_month
tmp['dom_bucket'] = pd.cut(tmp['dom_ratio'], bins=[0, 0.33, 0.66, 1.0], labels=['early(초순)', 'mid(중순)', 'late(말일)'])

profile = tmp.groupby('dom_bucket', observed=True)[['ac_trgt_daily', 'co_trgt_daily', 'contract_cnt', 'delivery_cnt']].mean()
print(profile.round(3))
print()
print('late/early 비율 (>1 이면 월말 쏠림):')
print((profile.loc['late(말일)'] / profile.loc['early(초순)']).round(2))

### 5-1. 결론 (2026-07-07 스냅샷 기준 실측 결과)

`om_contract.csv`/`sales_target.csv` 스냅샷으로 위 로직을 미리 검증한 결과, **당초 가설(AC=계약, CO=출고)과 반대**로 나왔다:

- **월 합계 비교는 무의미**: `contract_cnt`와 `delivery_cnt`의 상관계수가 0.98로 사실상 같은 신호라서, 월 단위로는 어느 타겟이 무엇인지 구분이 안 됨.
- **일별 시차(lag) 스캔**: 전사 합계 기준 `ac_trgt_daily`는 lag=0(당일)에서 `delivery_cnt`와 상관계수 **0.42**로 뚜렷한 피크(다른 모든 lag는 ±0.15 이내 잡음 수준). 반면 `co_trgt_daily`는 lag=0에서 `contract_cnt`와 **0.25**로 그 행에서는 최댓값이지만 피크가 상대적으로 약함.
- **월중 쏠림(day-of-month) 패턴**: `delivery_cnt`(late/early=1.31)와 `ac_trgt_daily`(3.62)는 둘 다 월말에 몰리는 전형적인 "월말 출고 밀어내기" 곡선을 보이는 반면, `contract_cnt`(0.79)와 `co_trgt_daily`(0.75)는 둘 다 월말 쏠림이 없고 오히려 초순이 더 크거나 평탄함.

세 가지 독립적인 증거(피크 상관, 시차 방향, 월중 형태)가 모두 같은 방향을 가리킨다:

> **`ac_trgt` / `ac_trgt_daily` = 출고(delivery) 타겟, `co_trgt` / `co_trgt_daily` = 계약(contract) 타겟**

`FCT_SALES_TARGET_DAILY`를 라벨/피처로 쓸 때는 이 대응관계로 사용해야 하며, ML 설계서에 적힌 "AC=계약, CO=출고"는 반대로 정정이 필요하다.
(참고: `ac_trgt`가 월초 며칠간 NULL로 비어 있는 이유도 이걸로 설명된다 — 계약과 달리 출고는 계약 후 며칠 뒤에나 발생하므로, 월초에는 아직 출고 누적치가 0/NULL인 날이 이어지는 것.)